In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

In [65]:
!pip install -q langchain-openai langchain-core requests langchain-huggingface

## 1.library Load

In [66]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

## 2.Tool Creation

In [77]:
import requests
from typing import Annotated
from langchain_core.tools import InjectedToolArg, tool

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> dict:
    """Fetches the currency conversion factor between a base currency and a target currency."""
    url = f"https://v6.exchangerate-api.com/v6/b15a3357d0ad002f1cb34623/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """Calculates the target currency value from a given base currency value using an injected conversion rate."""
    return base_currency_value * conversion_rate


In [78]:
ConversionAPI=get_conversion_factor.invoke({"base_currency":"USD", "target_currency":"INR"})
ConversionAPI

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1790812801,
 'time_last_update_utc': 'Thu, 01 Oct 2026 00:00:01 +0000',
 'time_next_update_unix': 1790899201,
 'time_next_update_utc': 'Fri, 02 Oct 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 96.0159}

In [79]:
convert.invoke({"base_currency_value": 10, "conversion_rate": ConversionAPI["conversion_rate"]})

960.159

## 3.Tool Binding


In [80]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

In [81]:
llm_with_tool = model.bind_tools([get_conversion_factor, convert])

In [82]:
message = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [83]:
message

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [103]:
ai_message = llm_with_tool.invoke(message)
ai_message

AIMessage(content='To answer your original question, the conversion factor between INR and USD is 0.01041. Based on this conversion factor, 10 INR is equivalent to 0.1041 USD.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 573, 'total_tokens': 615}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f8c6-e0ca-76b0-9144-9fe81286a6cf-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 573, 'output_tokens': 42, 'total_tokens': 615})

In [86]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'call_Nf3zjUTXGGPvaZJoZ6vqWLW1',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_efKp8PaI9pM9GKSFa6bPY1MW',
  'type': 'tool_call'}]

In [90]:
message.append(ai_message)

In [91]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    message.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    message.append(tool_message2)



In [93]:
message

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "INR", "target_currency": "USD"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'call_Nf3zjUTXGGPvaZJoZ6vqWLW1', 'type': 'function'}, {'function': {'arguments': '{"base_currency_value": 10}', 'name': 'convert', 'description': None}, 'id': 'call_efKp8PaI9pM9GKSFa6bPY1MW', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 366, 'total_tokens': 404}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0f8bb-c2b5-7ff3-856f-692904a92a78-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'INR', 'target_currency': 'USD'}, 'id': 'call_Nf3zjUTXGGPvaZ

## 4.Tool Execution


In [97]:
llm_with_tool.invoke(message).content

'The conversion factor between INR and USD is 0.01041, and based on that, 10 INR is equivalent to 0.1041 USD.'

## Step 5: Initialize the Agent

In [108]:
from langchain.agents import create_agent

# Step 5: Initialize the Agent

agent_executor = create_agent(
    model=model,
    tools=[get_conversion_factor, convert]
)

## Step 6: Run the Agent

In [120]:
user_query = "Hi how are you?"
response = agent_executor.invoke({"messages": [{"role": "user", "content": user_query}]})
response

{'messages': [HumanMessage(content='Hi how are you?', additional_kwargs={}, response_metadata={}, id='703c1374-a245-42dd-afb0-e325a18f88bb'),
  AIMessage(content="I'm functioning properly, thank you for asking. I'm ready to assist you with the available functions. What would you like to do first?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 346, 'total_tokens': 376}, 'model_name': 'meta-llama/Llama-3.1-8B-Instruct', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0f8d3-7e9b-70f1-891c-4760f6fdea88-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 346, 'output_tokens': 30, 'total_tokens': 376})]}

In [119]:
print(response["messages"][-1].content)

I'm functioning properly, thank you for asking.
